# Pooled Brain-State Analysis — Castro-Style
### TVBToolkit · Whole-Brain Simulations

**Standalone notebook** — runs independently of the full analysis pipeline.

Implements the Castro et al. (2024) pooled phase-coherence brain-state method:

- Pool all subjects' phase-coherence vectors → one shared k-means
- Sort centroids by SC-FC coupling (SFC) against a reference SC
- Per-subject occupancy in the shared state space
- Two SFC variants: **pooled-ref SC** and **subject-specific SC** (personalised)
- Castro-style violin plots + SFC vs occupancy regression scatter

**Parallelisation** — phase-pattern extraction runs concurrently across all subjects using `ThreadPoolExecutor` (scipy/numpy release the GIL so threading gives real speedup). Rate and BOLD k-means run simultaneously. All stages have a live progress bar.

> **Expected finding:** CNT occupies low-SFC states (complex dynamics); COMA/UWS occupies high-SFC states (anatomy-constrained). Slope of occupancy–SFC regression should be *negative* for CNT, *positive* for COMA/UWS.

In [1]:
from __future__ import annotations

import os, sys, time, threading, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as _mpatches

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ── TVBToolkit ────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-cache')
os.environ.setdefault('TVB_USER_HOME', str(PROJECT_ROOT / '.tvb-temp'))

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from tvbtoolkit.analysis.brain_states import (
    phase_patterns, cluster_brain_states, _compute_occupancy,
)
from scipy.stats import pearsonr as _pearsonr
from scipy.stats import mannwhitneyu as _mwu
from scipy import stats as _stats

# ── IPython progress display (built-in to every Jupyter kernel) ───────────────
try:
    from IPython.display import display as _ipy_display, HTML as _HTML
    _IN_NOTEBOOK = True
except ImportError:
    _IN_NOTEBOOK = False

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 7, 'axes.labelsize': 8, 'axes.titlesize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'axes.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'xtick.major.size': 3, 'ytick.major.size': 3,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'lines.linewidth': 1.0,
    'figure.dpi': 150, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.05,
})

print(f'Project root : {PROJECT_ROOT}')
print(f'CPU count    : {os.cpu_count()}')
print('Imports OK.')

Project root : /Users/borjan/CNRS/projects/TVBToolkit
CPU count    : 12
Imports OK.


---
## Progress Bar Utility

A lightweight inline progress bar built on `IPython.display` — no external packages needed.

In [2]:
class ProgressBar:
    """Thread-safe inline HTML progress bar for Jupyter notebooks.

    Falls back to plain print in non-notebook environments.
    """
    # Wes Anderson palette matched to notebook theme
    _COLOURS = {
        'green':  '#5B8A72',
        'blue':   '#3B4A6B',
        'mauve':  '#8B6B8B',
        'sienna': '#C5622F',
        'amber':  '#E8B56D',
    }

    def __init__(self, total: int, desc: str = '', colour: str = 'green'):
        self.total   = max(int(total), 1)
        self.desc    = desc
        self._n      = 0
        self._lock   = threading.Lock()
        self._t0     = time.perf_counter()
        self._colour = self._COLOURS.get(colour, colour)
        self._errors = 0
        if _IN_NOTEBOOK:
            self._handle = _ipy_display(_HTML(''), display_id=True)
        self._render()

    # ── Public API ──────────────────────────────────────────────────────────
    def update(self, n: int = 1, *, error: bool = False) -> None:
        """Advance the bar by `n` steps."""
        with self._lock:
            self._n = min(self._n + n, self.total)
            if error:
                self._errors += 1
        self._render()

    def done(self, msg: str = '') -> None:
        """Mark the bar as complete with an optional message."""
        with self._lock:
            self._n = self.total
        elapsed = time.perf_counter() - self._t0
        suffix  = msg or f'complete in {elapsed:.1f}s'
        err_txt = f' | {self._errors} errors' if self._errors else ''
        if _IN_NOTEBOOK:
            self._handle.update(_HTML(
                f'<div style="font-family:monospace; font-size:12px; '
                f'color:{self._colour}; margin:2px 0">'
                f'&#10003; <b>{self.desc}</b> — {suffix}{err_txt}</div>'
            ))
        else:
            print(f'[DONE] {self.desc} — {suffix}{err_txt}')

    # ── Rendering ───────────────────────────────────────────────────────────
    def _render(self) -> None:
        pct     = int(100 * self._n / self.total)
        elapsed = time.perf_counter() - self._t0
        rate    = self._n / elapsed if elapsed > 0.001 else 0
        eta     = (self.total - self._n) / rate if rate > 0 else 0
        err_txt = (
            f' &nbsp;<span style="color:#c0392b">⚠ {self._errors} failed</span>'
            if self._errors else ''
        )
        bar_html = (
            f'<div style="font-family:monospace; font-size:12px; margin:4px 0">'
            f'  <span style="color:#444"><b>{self.desc}</b></span>'
            f'  &nbsp;<span style="color:#888">{self._n}/{self.total}</span>'
            f'  {err_txt}'
            f'  <div style="background:#e8e8e8; border-radius:6px; height:12px; '
            f'              width:420px; margin:4px 0; overflow:hidden">'
            f'    <div style="background:{self._colour}; width:{pct}%; height:12px; '
            f'               border-radius:6px"></div>'
            f'  </div>'
            f'  <span style="color:#999; font-size:10px">'
            f'    {pct}% &nbsp;|&nbsp; {elapsed:.1f}s elapsed'
            f'    {" &nbsp;|&nbsp; ETA " + f"{eta:.0f}s" if self._n < self.total and rate > 0 else ""}'
            f'  </span>'
            f'</div>'
        )
        if _IN_NOTEBOOK:
            self._handle.update(_HTML(bar_html))
        else:
            filled = int(30 * self._n / self.total)
            bar    = '█' * filled + '░' * (30 - filled)
            print(f'\r{self.desc} |{bar}| {pct}% {self._n}/{self.total}', end='', flush=True)
            if self._n >= self.total:
                print()

print('ProgressBar ready.')

ProgressBar ready.


---
## 1. Configuration

In [3]:
# ═══════════════════════════════════════════════════════════════════════════
#  CONFIGURATION  —  edit these to match your data
# ═══════════════════════════════════════════════════════════════════════════

ROUTE_SELECT    = 'private_only'
SCENARIO_FILTER = None

SIM_DIR      = PROJECT_ROOT / 'notebooks' / 'outputs' / 'ba_sim_v2' / ROUTE_SELECT / 'sims'
DATASET_ROOT = PROJECT_ROOT / 'data' / 'doc_patients_new_data' / 'converted_structural'
FIG_DIR      = PROJECT_ROOT / 'notebooks' / 'outputs' / 'pooled_brain_states' / 'figs'
FIG_DIR.mkdir(parents=True, exist_ok=True)

COHORT_TO_CONDITION = {
    'control': 'CNT', 'emcs': 'EMCS', 'mcs': 'MCS',
    'uws': 'UWS', 'coma': 'COMA',
}
CONDITION_ORDER = ['COMA', 'UWS', 'MCS', 'EMCS', 'CNT']
CONDITIONS      = CONDITION_ORDER

COND_COLORS = {
    'COMA': '#3B4A6B', 'UWS': '#8B6B8B', 'MCS': '#C5622F',
    'EMCS': '#E8B56D', 'CNT': '#5B8A72',
}
STATE_PALETTE = ['#3B4A6B', '#8B6B8B', '#C5622F', '#E8B56D', '#5B8A72']

N_STATES           = 5
RANDOM_SEED_KMEANS = 42
N_INIT_KMEANS      = 10
MAX_ITER_KMEANS    = 300

DT_MS             = 5.0
TRANSIENT_MS      = 0.0
BANDPASS_RATE_HZ  = (2.0, 80.0)
FILTER_ORDER_RATE = 4
TR_SECONDS        = 2.4
BANDPASS_BOLD_HZ  = (0.01, 0.20)
FILTER_ORDER_BOLD = 3

N_SUBJECTS_MAX = None

# File I/O uses a thread pool (purely I/O-bound, no BLAS contention).
N_WORKERS = min(os.cpu_count() or 4, 8)

# ── Per-subject watchdog timeout (seconds) ───────────────────────────────
# Phase extraction runs sequentially.  A healthy subject at 200 Hz processes
# in < 30 s.  Some subjects carry pathological data (near-zero variance, NaN
# propagation, or an array length that triggers a slow FFT path) that makes
# scipy hang indefinitely.  The watchdog kills only those subjects; all others
# always complete.  120 s (2 min) is 4× the worst-case healthy subject.
# Increase if your simulations are unusually long.
SUBJECT_TIMEOUT_S = 120

print('Configuration loaded.')
print(f'  SIM_DIR          : {SIM_DIR}')
print(f'  FIG_DIR          : {FIG_DIR}')
print(f'  N_WORKERS (I/O)  : {N_WORKERS}')
print(f'  SUBJECT_TIMEOUT_S: {SUBJECT_TIMEOUT_S} s  '
      f'(only fires on genuinely hung subjects)')


Configuration loaded.
  SIM_DIR          : /Users/borjan/CNRS/projects/TVBToolkit/notebooks/outputs/ba_sim_v2/private_only/sims
  FIG_DIR          : /Users/borjan/CNRS/projects/TVBToolkit/notebooks/outputs/pooled_brain_states/figs
  N_WORKERS (I/O)  : 8
  SUBJECT_TIMEOUT_S: 120 s  (only fires on genuinely hung subjects)


---
## 2. Data Loading

Loads simulations from `ba_sim_v2`. Falls back to synthetic data automatically if the directory is empty.

In [4]:
try:
    from tvbtoolkit.datasets.brain_act import load_subject_structural
    _HAS_TVBTOOLKIT = True
except ImportError:
    _HAS_TVBTOOLKIT = False


def _load_sc(cohort: str, subject_id: str) -> np.ndarray:
    if not _HAS_TVBTOOLKIT or not DATASET_ROOT.exists():
        return np.eye(10)
    try:
        c, l, _, _ = load_subject_structural(
            subject_id=subject_id, cohort=cohort,
            dataset_root=DATASET_ROOT, validate=True,
            enforce_symmetry=True, zero_diagonal=True, nonfinite='raise',
        )
        c, l = np.asarray(c, float), np.asarray(l, float)
        np.fill_diagonal(c, 0.0); np.fill_diagonal(l, 0.0)
        if cohort.lower() in {'coma', 'uws', 'mcs', 'emcs'}:
            l[(c == 0.0) & (l != 0.0)] = 0.0
        mx = float(np.max(c))
        return c / mx if mx > 0 else c
    except Exception as exc:
        return np.eye(10)


def _load_from_sim_dir(sim_dir: Path) -> dict:
    data_out = {c: [] for c in CONDITION_ORDER}
    if not sim_dir.exists():
        return data_out
    npz_tasks = []   # (scenario, cohort, cond, subject_id, npz_path)
    for scen in sorted(sim_dir.iterdir()):
        if not scen.is_dir() or (SCENARIO_FILTER and scen.name not in SCENARIO_FILTER):
            continue
        for chort in sorted(scen.iterdir()):
            if not chort.is_dir():
                continue
            cond = COHORT_TO_CONDITION.get(chort.name)
            if cond is None:
                continue
            for subj_path in sorted(chort.iterdir()):
                if not subj_path.is_dir():
                    continue
                if N_SUBJECTS_MAX and len(data_out[cond]) >= N_SUBJECTS_MAX:
                    continue
                npz_files = sorted(subj_path.glob('seed_*.npz'))
                if npz_files:
                    npz_tasks.append((scen.name, chort.name, cond,
                                      subj_path.name, npz_files[0].resolve()))

    if not npz_tasks:
        return data_out

    # ── Parallel NPZ load ──────────────────────────────────────────────────
    def _load_one_npz(task):
        scen_key, cohort, cond, subject_id, npz_path = task
        try:
            d        = np.load(npz_path, allow_pickle=True)
            rate     = np.asarray(d['rate'], float) * 1e3   # kHz → Hz
            rate_t   = np.asarray(d['time_rate_ms'], float)
            bold_t   = np.asarray(d['time_bold_ms'], float) if 'time_bold_ms' in d else np.array([])
            bold     = np.asarray(d['bold'],         float) if 'bold'         in d else np.empty((0, rate.shape[1]))
            sc       = _load_sc(cohort, subject_id)
            return dict(cond=cond, cohort=cohort, subject_id=subject_id,
                        scenario_key=scen_key, rate=rate, rate_time=rate_t,
                        bold=bold, bold_time=bold_t, sc=sc)
        except Exception as exc:
            return dict(error=str(exc), cond=cond)

    pb = ProgressBar(len(npz_tasks), desc='Loading simulation files', colour='blue')
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = {ex.submit(_load_one_npz, t): t for t in npz_tasks}
        for fut in as_completed(futs):
            res = fut.result()
            if 'error' not in res:
                data_out[res['cond']].append(res)
            pb.update(error='error' in res)
    pb.done()
    return data_out


def _generate_synthetic(cond, n_regions=10, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    n_bins = int(30_000 / DT_MS)
    t = np.arange(n_bins) * DT_MS
    p = dict(CNT=(40,8,2,0.6,5), EMCS=(30,5.5,2.2,0.5,4),
             MCS=(20,3.5,2.5,0.3,3), UWS=(5,0.8,3,0.1,1.5),
             COMA=(3,0.5,3.5,0.05,1)).get(cond,(20,3,2,0.3,3))
    f_osc, amp, noise, coupling, base = p
    sc = rng.uniform(0,1,(n_regions,n_regions))
    sc = (sc+sc.T)/2; sc[sc<0.5]=0; np.fill_diagonal(sc,0)
    mx = sc.max(); sc = sc/mx if mx>0 else sc
    omega = 2*np.pi*f_osc/1000
    phases = rng.uniform(0, 2*np.pi, n_regions)
    rate = np.column_stack([
        np.maximum(0, base + amp*np.sin(omega*t+phases[i])
                   + coupling*amp*np.sin(0.3*omega*t)
                   + noise*rng.standard_normal(n_bins))
        for i in range(n_regions)
    ])
    stride = max(1, int(TR_SECONDS*1000/DT_MS))
    idx = np.arange(0, n_bins, stride)
    kern = np.ones(min(stride*2, n_bins//2)) / min(stride*2, n_bins//2)
    bold = np.apply_along_axis(lambda x: np.convolve(x,kern,'same'),0,rate)[idx]
    bold = (bold-bold.mean(0))/(bold.std(0)+1e-9)
    return dict(rate=rate, rate_time=t, bold=bold, bold_time=t[idx],
                sc=sc, condition=cond, cohort=cond.lower(),
                subject_id='synthetic', synthetic=True)


data = _load_from_sim_dir(SIM_DIR)
CONDITIONS = [c for c in CONDITION_ORDER if data.get(c)]

if not CONDITIONS:
    print('No simulation data found — generating synthetic fallback.')
    data = {c: [] for c in CONDITION_ORDER}
    pb_syn = ProgressBar(15, desc='Generating synthetic data', colour='sienna')
    for cond in ['COMA', 'UWS', 'MCS', 'EMCS', 'CNT']:
        for s in range(3):
            rng_s = np.random.default_rng(abs(hash(cond+str(s))) % 2**31)
            data[cond].append(_generate_synthetic(cond, rng=rng_s))
            pb_syn.update()
    pb_syn.done()
    CONDITIONS = [c for c in CONDITION_ORDER if data.get(c)]

print(f'Conditions: {CONDITIONS}')
total_subj = sum(len(data[c]) for c in CONDITIONS)
print(f'Total subjects: {total_subj}')
for cond in CONDITIONS:
    ex = data[cond][0]
    print(f'  {cond:5s}: {len(data[cond])} subj | '
          f'rate {np.asarray(ex["rate"]).shape} | '
          f'bold {np.asarray(ex["bold"]).shape}')

Conditions: ['COMA', 'UWS', 'MCS', 'EMCS', 'CNT']
Total subjects: 189
  COMA : 10 subj | rate (23600, 90) | bold (49, 90)
  UWS  : 51 subj | rate (23600, 90) | bold (49, 90)
  MCS  : 75 subj | rate (23600, 90) | bold (49, 90)
  EMCS : 18 subj | rate (23600, 90) | bold (49, 90)
  CNT  : 35 subj | rate (23600, 90) | bold (49, 90)


In [ ]:
def _safe_pearson(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 3 or np.std(a[mask]) < 1e-12 or np.std(b[mask]) < 1e-12:
        return np.nan
    return float(_pearsonr(a[mask], b[mask])[0])


def _call_with_watchdog(fn, *args, timeout_s=SUBJECT_TIMEOUT_S):
    """Call fn(*args) in a daemon thread; return (result, None) on success
    or (None, reason_str) if fn raises or exceeds timeout_s seconds.

    The daemon thread cannot be force-killed — if it times out it keeps
    running silently in the background — but the main loop moves on.
    At 300 s the watchdog only fires on genuinely hung scipy calls.
    """
    _res, _exc, _done = [None], [None], threading.Event()
    def _target():
        try:
            _res[0] = fn(*args)
        except Exception as e:
            _exc[0] = str(e)
        finally:
            _done.set()
    threading.Thread(target=_target, daemon=True).start()
    if _done.wait(timeout=timeout_s):
        return (_res[0], None) if _exc[0] is None else (None, _exc[0])
    return None, f'watchdog fired after {timeout_s}s — scipy call did not return'


print('Helpers loaded.')


Helpers loaded.


: 

---
## 3. Phase-Pattern Extraction + Pooled k-Means

### 3a — Firing Rates

Subjects are processed **sequentially** with a per-subject timeout. This is intentional: `scipy.signal.filtfilt` and `scipy.signal.hilbert` already use BLAS/LAPACK threads internally. Running N outer Python threads \xd7 M inner BLAS threads on a machine with fewer cores causes all threads to deadlock waiting for CPU time — the notebook stalls mid-run.

Sequential Python + multi-threaded scipy is the correct pattern. Each subject still benefits from scipy's internal parallelism; no subjects contend for the same threads.

A daemon-thread timeout (`SUBJECT_TIMEOUT_S`) ensures that if any subject's data causes an unexpectedly slow computation, it is skipped and flagged rather than blocking the rest of the run.

In [ ]:
# ── §3a  Phase extraction — firing rates (sequential + watchdog) ────────────
#
# Sequential: one subject at a time so scipy's internal BLAS threads never
# contend with each other (avoids the thread-pool deadlock seen previously).
#
# Watchdog: a daemon thread runs each phase_patterns() call.  If it doesn't
# return within SUBJECT_TIMEOUT_S seconds the main loop logs the subject and
# moves on.  At 300 s this only fires on genuinely hung scipy calls — all
# healthy subjects finish well under that limit.

_tasks_rate = [
    (cond, s, subj)
    for cond in CONDITIONS
    for s, subj in enumerate(data[cond])
]

pool_patterns_rate = []
pool_meta_rate     = []
sc_all_rate        = []
_failed_rate       = []

pb_rate = ProgressBar(len(_tasks_rate),
                      desc='Phase extraction — firing rates',
                      colour='green')

for cond, s, subj in _tasks_rate:
    rate = np.asarray(subj['rate'], float)
    sc   = np.asarray(subj['sc'],   float)

    def _do_extract_rate(r=rate):
        return phase_patterns(
            r,
            pipeline='firing_rate',
            bandpass_hz=BANDPASS_RATE_HZ,
            filter_order=FILTER_ORDER_RATE,
            dt_ms=DT_MS,
            transient_ms=TRANSIENT_MS,
        )

    ok = False
    result, err = _call_with_watchdog(_do_extract_rate)
    if err:
        _failed_rate.append((cond, s, err))
    else:
        pats, _, _, _ = result
        if pats.shape[0] >= N_STATES:
            n_reg  = sc.shape[0]
            sc_vec = sc[np.triu_indices(n_reg, k=1)]
            pool_patterns_rate.append(pats)
            pool_meta_rate.append(dict(
                cond=cond, subj=s,
                sc_vec=sc_vec, n_rows=pats.shape[0]
            ))
            sc_all_rate.append(sc)
            ok = True
        else:
            _failed_rate.append((cond, s,
                f'only {pats.shape[0]} time points after filtering'))
    pb_rate.update(error=not ok)

pb_rate.done(f'{len(pool_meta_rate)}/{len(_tasks_rate)} subjects')

if _failed_rate:
    print(f'  {len(_failed_rate)} subject(s) failed / timed out:')
    for cond, s, reason in _failed_rate:
        print(f'    [{cond} s{s}] {reason}')

X_rate     = np.vstack(pool_patterns_rate).astype(np.float32)
sc_ref_mat = np.mean(np.stack(sc_all_rate, axis=0), axis=0)
n_reg_rate = sc_ref_mat.shape[0]
sc_ref_vec = sc_ref_mat[np.triu_indices(n_reg_rate, k=1)]
print(f'Pooled rate matrix : {X_rate.shape}')


### 3b — BOLD

Same extraction for the BOLD signal.  Subjects with fewer than 30 TRs are silently skipped.

In [ ]:
# ── §3b  Phase extraction — BOLD (sequential + watchdog) ───────────────────

_tasks_bold = [
    (cond, s, subj)
    for cond in CONDITIONS
    for s, subj in enumerate(data[cond])
    if np.asarray(subj['bold']).shape[0] >= 30
]

pool_patterns_bold = []
pool_meta_bold     = []
sc_all_bold        = []

if _tasks_bold:
    _failed_bold = []
    pb_bold = ProgressBar(len(_tasks_bold),
                          desc='Phase extraction — BOLD',
                          colour='mauve')
    for cond, s, subj in _tasks_bold:
        bold = np.asarray(subj['bold'], float)
        sc   = np.asarray(subj['sc'],   float)

        def _do_extract_bold(b=bold):
            return phase_patterns(
                b,
                pipeline='brain_act_legacy',
                tr_seconds=TR_SECONDS,
                bandpass_hz=BANDPASS_BOLD_HZ,
                filter_order=FILTER_ORDER_BOLD,
            )

        ok = False
        result, err = _call_with_watchdog(_do_extract_bold)
        if err:
            _failed_bold.append((cond, s, err))
        else:
            pats, _, _, _ = result
            if pats.shape[0] >= N_STATES:
                n_reg  = sc.shape[0]
                sc_vec = sc[np.triu_indices(n_reg, k=1)]
                pool_patterns_bold.append(pats)
                pool_meta_bold.append(dict(
                    cond=cond, subj=s,
                    sc_vec=sc_vec, n_rows=pats.shape[0]
                ))
                sc_all_bold.append(sc)
                ok = True
            else:
                _failed_bold.append((cond, s,
                    f'only {pats.shape[0]} time points after filtering'))
        pb_bold.update(error=not ok)

    pb_bold.done(f'{len(pool_meta_bold)}/{len(_tasks_bold)} subjects')
    if _failed_bold:
        print(f'  {len(_failed_bold)} subject(s) failed / timed out:')
        for cond, s, reason in _failed_bold:
            print(f'    [{cond} s{s}] {reason}')
    X_bold = (
        np.vstack(pool_patterns_bold).astype(np.float32)
        if len(pool_patterns_bold) >= N_STATES else None
    )
else:
    print('No subjects with sufficient BOLD data — BOLD analysis skipped.')
    X_bold = None

if X_bold is not None:
    sc_ref_mat_bold = np.mean(np.stack(sc_all_bold, axis=0), axis=0)
    sc_ref_vec_bold = sc_ref_mat_bold[
        np.triu_indices(sc_ref_mat_bold.shape[0], k=1)
    ]
    print(f'Pooled BOLD matrix : {X_bold.shape}')
else:
    sc_ref_mat_bold = sc_ref_vec_bold = None


### 3c — Pooled k-Means (Rate + BOLD in Parallel)

Both k-means runs are dispatched simultaneously to separate threads. sklearn's KMeans releases the GIL for its core BLAS operations so both runs proceed concurrently.

In [ ]:
# ── §3c  Parallel k-means — rate and BOLD simultaneously ─────────────────────

_kmeans_results = {}
_kmeans_done    = threading.Event()

def _run_kmeans(key, X, sc_ref_vec_local):
    """Run pooled k-means + SFC sort; stores result in _kmeans_results[key]."""
    labels_raw, centers_raw = cluster_brain_states(
        X, n_states=N_STATES,
        random_seed=RANDOM_SEED_KMEANS,
        n_init=N_INIT_KMEANS,
        max_iter=MAX_ITER_KMEANS,
    )
    sfc_raw    = np.array([_safe_pearson(c, sc_ref_vec_local) for c in centers_raw])
    sort_order = np.argsort(np.nan_to_num(sfc_raw, nan=np.inf))
    centers    = centers_raw[sort_order]
    sfc        = sfc_raw[sort_order]
    remap      = np.empty_like(sort_order)
    remap[sort_order] = np.arange(len(sort_order))
    labels     = remap[labels_raw]
    _kmeans_results[key] = dict(labels=labels, centers=centers, sfc=sfc)


# Progress bar tracks both jobs finishing
n_kmeans = 1 + (1 if X_bold is not None else 0)
pb_km = ProgressBar(n_kmeans, desc='Pooled k-means', colour='sienna')

def _kmeans_worker(key, X, sc_vec):
    _run_kmeans(key, X, sc_vec)
    pb_km.update()

with ThreadPoolExecutor(max_workers=2) as ex:
    futs_km = [ex.submit(_kmeans_worker, 'rate', X_rate, sc_ref_vec)]
    if X_bold is not None:
        futs_km.append(ex.submit(_kmeans_worker, 'bold', X_bold, sc_ref_vec_bold))
    for f in as_completed(futs_km):
        f.result()   # re-raise any exception

pb_km.done()

# Unpack results
labels_rate  = _kmeans_results['rate']['labels']
centers_rate = _kmeans_results['rate']['centers']
sfc_rate     = _kmeans_results['rate']['sfc']

if X_bold is not None:
    labels_bold  = _kmeans_results['bold']['labels']
    centers_bold = _kmeans_results['bold']['centers']
    sfc_bold     = _kmeans_results['bold']['sfc']
else:
    labels_bold = centers_bold = sfc_bold = None

print(f'Rate  SFC (low→high): {np.round(sfc_rate, 3)}')
if sfc_bold is not None:
    print(f'BOLD  SFC (low→high): {np.round(sfc_bold, 3)}')

---
## 4. Per-Subject Occupancy and Subject-Specific SFC

For each subject, slice their global label array to get occupancy in the shared state space, and correlate each centroid against the subject's own SC for the personalised SFC.

In [ ]:
# ── §4  Per-subject occupancy + subject-specific SFC (sequential) ────────────
# Occupancy is just array slicing + bincount — negligible CPU time per subject.

def build_pooled_records(pool_meta, labels_sorted, centers_sorted,
                          sfc_ref_sorted, tag=''):
    records = []
    ptr     = 0
    pb = ProgressBar(len(pool_meta),
                     desc=f'Occupancy + subject SFC — {tag}',
                     colour='amber')
    for meta in pool_meta:
        lab_slice = labels_sorted[ptr : ptr + meta['n_rows']]
        ptr      += meta['n_rows']
        occ       = _compute_occupancy(lab_slice, N_STATES)
        sfc_sub   = np.array([_safe_pearson(c, meta['sc_vec'])
                               for c in centers_sorted])
        records.append(dict(
            cond=meta['cond'], subj=meta['subj'],
            occupancy=occ,
            sfc_ref=sfc_ref_sorted.copy(),
            sfc_sub=sfc_sub,
        ))
        pb.update()
    pb.done()
    return records


records_rate = build_pooled_records(
    pool_meta_rate, labels_rate, centers_rate, sfc_rate, tag='rate'
)

records_bold = (
    build_pooled_records(
        pool_meta_bold, labels_bold, centers_bold, sfc_bold, tag='BOLD'
    ) if X_bold is not None else []
)

example = next((r for r in records_rate if r['cond'] == CONDITIONS[-1]),
               records_rate[0])
print(f"\nExample — {example['cond']} s{example['subj']}")
print(f"  Occupancy : {np.round(example['occupancy']*100, 1)} %")
print(f"  SFC(ref)  : {np.round(example['sfc_ref'], 3)}")
print(f"  SFC(sub)  : {np.round(example['sfc_sub'], 3)}")


---
## 5. Castro-Style Violin Plots

One panel per shared state (S0 → S4, low → high SFC).  
Mann-Whitney U tests compare each DoC group against CNT.

In [ ]:
# ── §5  Violin plots ─────────────────────────────────────────────────────────

def violin_plot(records, sfc_ref, title_prefix='', fig_path=None,
                n_states=N_STATES):
    if not records:
        print(f'No records — skipping: {title_prefix}'); return

    K = n_states
    fig, axes = plt.subplots(1, K, figsize=(2.4*K, 3.8), sharey=False)
    if K == 1:
        axes = [axes]

    for k, ax in enumerate(axes):
        cond_data = [
            np.array([r['occupancy'][k]*100 for r in records if r['cond'] == c])
            for c in CONDITIONS
        ]
        positions = np.arange(len(CONDITIONS))
        vp = ax.violinplot(
            [d if len(d) > 0 else [0.0] for d in cond_data],
            positions=positions, showmedians=True,
            showextrema=False, widths=0.6,
        )
        for body, cond in zip(vp['bodies'], CONDITIONS):
            body.set_facecolor(COND_COLORS.get(cond, '#aaa'))
            body.set_alpha(0.75); body.set_edgecolor('black'); body.set_linewidth(0.6)
        if 'cmedians' in vp:
            vp['cmedians'].set_color('black'); vp['cmedians'].set_linewidth(1.5)

        rng_j = np.random.default_rng(7)
        for xi, (vals, cond) in enumerate(zip(cond_data, CONDITIONS)):
            if not len(vals): continue
            jit = rng_j.uniform(-0.12, 0.12, size=len(vals))
            ax.scatter(xi+jit, vals, color=COND_COLORS.get(cond,'#aaa'),
                       s=14, alpha=0.85, edgecolors='white',
                       linewidths=0.3, zorder=4)

        cnt_idx  = CONDITIONS.index('CNT') if 'CNT' in CONDITIONS else -1
        cnt_vals = cond_data[cnt_idx] if cnt_idx >= 0 else np.array([])
        sig_pairs = []
        for xi, cond in enumerate(CONDITIONS):
            if cond=='CNT' or not len(cond_data[xi]) or not len(cnt_vals): continue
            _, p = _mwu(cnt_vals, cond_data[xi], alternative='two-sided')
            stars = ('***' if p<0.001 else '**' if p<0.01
                     else '*' if p<0.05 else 'ns')
            if stars != 'ns':
                sig_pairs.append((cnt_idx, xi, stars))

        ax.autoscale()
        y_top = ax.get_ylim()[1]
        for bi, (xi1, xi2, stars) in enumerate(sig_pairs):
            yb = y_top*(1.07 + 0.10*bi)
            ax.plot([xi1,xi1,xi2,xi2],[yb*.99,yb,yb,yb*.99],lw=0.8,color='black')
            ax.text((xi1+xi2)/2, yb*1.005, stars, ha='center', va='bottom', fontsize=6)
        ax.set_ylim(0, ax.get_ylim()[1]*(1.06 + 0.12*len(sig_pairs)))

        sfc_k = float(sfc_ref[k]) if k<len(sfc_ref) and np.isfinite(sfc_ref[k]) else float('nan')
        ax.set_xticks(positions)
        ax.set_xticklabels(CONDITIONS, fontsize=6, rotation=30, ha='right')
        ax.set_ylabel('Occupancy (%)' if k==0 else '', fontsize=7)
        ax.set_title(f'State S{k}\nSFC = {sfc_k:.3f}', fontsize=7, fontweight='bold')

    fig.suptitle(
        f'{title_prefix}\nPooled shared-centroid occupancy by condition '
        f'(sorted low\u2192high SC-FC)',
        fontsize=8, fontweight='bold'
    )
    fig.tight_layout()
    if fig_path:
        fig.savefig(fig_path, dpi=300, bbox_inches='tight')
        print(f'  Saved \u2192 {fig_path.name}')
    plt.show()


violin_plot(records_rate, sfc_ref=sfc_rate,
            title_prefix='A  Pooled Brain States — Firing Rates',
            fig_path=FIG_DIR / 'pooled_violin_rate.png')

if records_bold:
    violin_plot(records_bold, sfc_ref=sfc_bold,
                title_prefix='B  Pooled Brain States — BOLD',
                fig_path=FIG_DIR / 'pooled_violin_bold.png')
else:
    print('BOLD violin: skipped.')

---
## 6. SFC vs Occupancy Regression Scatter

Each point = one subject × state pair.  Per-condition regression + 95% CI band.  
Left column: pooled-ref SFC (same for all subjects per centroid).  
Right column: subject-specific SFC (personalised x-axis).

In [ ]:
# ── §6  SFC vs occupancy scatter ─────────────────────────────────────────────

def sfc_occ_scatter(records, sfc_mode='ref', ax=None,
                     title='', n_states=N_STATES):
    if not records: return
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(4.5, 3.5))

    for cond in CONDITIONS:
        xs, ys = [], []
        for r in records:
            if r['cond'] != cond: continue
            sv = r['sfc_ref'] if sfc_mode == 'ref' else r['sfc_sub']
            for k in range(n_states):
                if np.isfinite(sv[k]):
                    xs.append(sv[k]); ys.append(r['occupancy'][k]*100)
        if not xs: continue
        xs, ys = np.array(xs), np.array(ys)
        col = COND_COLORS.get(cond,'#aaa')
        ax.scatter(xs, ys, color=col, s=12, alpha=0.55,
                   edgecolors='white', linewidths=0.3, zorder=3)
        if len(xs) > 2:
            sl, ic, r_, p_, _ = _stats.linregress(xs, ys)
            xl = np.linspace(xs.min(), xs.max(), 80)
            sig = '***' if p_<0.001 else '**' if p_<0.01 else '*' if p_<0.05 else ''
            ax.plot(xl, sl*xl+ic, color=col, lw=1.8, zorder=4,
                    label=f'{cond}  r={r_:.2f}{sig}')
            n_ = len(xs)
            se_ = (np.sqrt(np.sum((ys-(sl*xs+ic))**2)/(n_-2))
                   * np.sqrt(1/n_ + (xl-xs.mean())**2/np.sum((xs-xs.mean())**2)))
            t_ = _stats.t.ppf(0.975, df=n_-2)
            ax.fill_between(xl, sl*xl+ic-t_*se_, sl*xl+ic+t_*se_,
                            color=col, alpha=0.10, zorder=2)

    xlab = ('SC-FC coupling — pooled-ref SC (Pearson r)'
            if sfc_mode=='ref'
            else 'SC-FC coupling — subject-specific SC (Pearson r)')
    ax.set_xlabel(xlab, fontsize=7)
    ax.set_ylabel('State occupancy (%)', fontsize=7)
    ax.set_title(title, fontsize=8, fontweight='bold', loc='left')
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=6, frameon=False,
              bbox_to_anchor=(1.01,1), loc='upper left',
              title='Condition', title_fontsize=6)
    if standalone:
        fig.tight_layout(); return fig


n_rows = 2 if records_bold else 1
fig, axes = plt.subplots(n_rows, 2, figsize=(8.0, 3.6*n_rows))
axes = np.array(axes).reshape(n_rows, 2)

sfc_occ_scatter(records_rate, sfc_mode='ref', ax=axes[0,0],
                title='A  Firing rate — pooled-ref SFC')
sfc_occ_scatter(records_rate, sfc_mode='sub', ax=axes[0,1],
                title='B  Firing rate — subject-specific SFC')
if records_bold:
    sfc_occ_scatter(records_bold, sfc_mode='ref', ax=axes[1,0],
                    title='C  BOLD — pooled-ref SFC')
    sfc_occ_scatter(records_bold, sfc_mode='sub', ax=axes[1,1],
                    title='D  BOLD — subject-specific SFC')

leg_h = [_mpatches.Patch(color=COND_COLORS[c], alpha=0.85, label=c)
         for c in CONDITIONS if c in COND_COLORS]
fig.legend(handles=leg_h, fontsize=6.5, frameon=False,
           loc='lower center', ncol=len(CONDITIONS),
           title='Condition', title_fontsize=6,
           bbox_to_anchor=(0.5, -0.04))
fig.suptitle('SFC vs Occupancy — Pooled Shared Centroids (Castro-style)',
             fontsize=9, fontweight='bold')
fig.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(FIG_DIR / 'pooled_sfc_vs_occ_scatter.png', dpi=300, bbox_inches='tight')
plt.show()
print('\u2713 Scatter saved.')

---
## 7. Shared Centroid Heatmaps  *(diagnostic)*

The K shared centroids plotted as region × region phase-coherence matrices, ordered S0 (most complex / anti-correlated) → S4 (most anatomy-like).

In [ ]:
# ── §7  Centroid heatmaps ─────────────────────────────────────────────────────

def plot_centroids(centers, sfc_vals, n_reg, title=''):
    K = centers.shape[0]
    fig, axes = plt.subplots(1, K, figsize=(2.5*K, 2.8))
    if K == 1: axes = [axes]
    iu, ju = np.triu_indices(n_reg, k=1)
    for k, ax in enumerate(axes):
        mat = np.zeros((n_reg, n_reg))
        mat[iu, ju] = centers[k]
        mat[ju, iu] = centers[k]
        im = ax.imshow(mat, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
        sfc_k = float(sfc_vals[k]) if np.isfinite(sfc_vals[k]) else float('nan')
        ax.set_title(f'S{k}  SFC={sfc_k:.3f}', fontsize=7, fontweight='bold')
        ax.axis('off')
    plt.colorbar(im, ax=axes[-1], fraction=0.046, pad=0.04, label='phase coherence')
    fig.suptitle(title, fontsize=8, fontweight='bold')
    fig.tight_layout(); plt.show()


plot_centroids(centers_rate, sfc_rate, n_reg_rate,
               'Shared centroids — Firing Rates (low\u2192high SFC)')

if X_bold is not None:
    n_reg_bold = sc_ref_mat_bold.shape[0]
    plot_centroids(centers_bold, sfc_bold, n_reg_bold,
                   'Shared centroids — BOLD (low\u2192high SFC)')